### ran on kaggle

In [ ]:
import json
import re
from html import unescape
from pathlib import Path

In [ ]:
NVALID_TOKENS = {"", "none", "null", "n/a", "na", "unknown", "-"}

def clean_text(value, decode_entities  = False) :
	"""Safely normalize text fields and optionally decode HTML entities."""
	if value is None:
		return ""
	text = str(value)
	if decode_entities:
		text = unescape(text)
	text=re.sub(r"\s+", " ", text).strip()
	return text


def parse_float(value) : ### get price without text
	
	if value is None:
		return None
	if isinstance(value, (int, float)):
		return float(value)

	text = clean_text(value).replace(",", "")
	if not text:
		return None

	match = re.search(r"[-+]?\d*\.?\d+", text)
	if not match:
		return None


	return float(match.group(0))

In [ ]:
def clean_product(product) :

	name = clean_text(product.get("name"))
	brand = clean_text(product.get("brand"))

	price = parse_float(product.get("price"))
	original_price = parse_float(product.get("original_price"))

	discount_raw = product.get("discount")
	discount = parse_float(discount_raw)

	price_value = price if price is not None else 0.0

	if discount_raw is None or clean_text(discount_raw) == "" or discount is None:
		discount_value = 0.0
		original_price_value = price_value
	else:
		discount_value = discount
		original_price_value = (
			original_price if original_price is not None else price_value
		)

	cleaned = {
		"product_id": clean_text(product.get("product_id")),
		"name": name,
		"brand": brand,
		"category": clean_text(product.get("category")),
		"price": float(price_value),
		"original_price": float(original_price_value),
		"discount": float(discount_value),
		"availability": clean_text(product.get("availability")),
		"description": clean_text(product.get("description")),
	}

	return cleaned

In [ ]:

input_path = Path("/kaggle/input/datasets/marwanelnabawy/big-data/crwal_results.json")
output_path =Path("cleaned_products2.json")

with input_path.open("r", encoding="utf-8") as file:
	payload = json.load(file).get("products", [])



cleaned_products = [clean_product(product) for product in payload]

with output_path.open("w", encoding="utf-8") as file:
	json.dump(cleaned_products, file, ensure_ascii=False, indent=2)